# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available online.

In [ ]:
# Install mlcroissant if not already installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL of the Croissant JSON-LD schema for the dataset
schema_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(schema_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review and explore the available record sets, fields, and their `@id`s.

In [ ]:
# List all the record sets and their fields by @id
record_sets = dataset.record_sets

print("Available Record Sets and their Fields:")
for rs_metadata in record_sets:
    print(f"- Record Set @id: {rs_metadata['@id']}, name: {rs_metadata.get('name','(no name found)')}")
    field_ids = [field['@id'] for field in rs_metadata.get('field', [])]
    print(f"  Fields: {field_ids}\n")

## 3. Data Extraction
Load the data from each record set into Pandas DataFrames. Record sets and fields are accessed via their `@id`.

In [ ]:
# Extract data from all record sets by @id
dfs = {}
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

for rs_id in record_set_ids:
    # Each record consists of {field_id: value, ...}
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            dfs[rs_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for Record Set: {rs_id}, shape: {dfs[rs_id].shape}")
        else:
            print(f"No records found for Record Set: {rs_id}")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# Show columns of the main data record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is not None and main_record_set_id in dfs:
    print(f"\nColumns in primary record set ({main_record_set_id}):")
    print(dfs[main_record_set_id].columns.tolist())
    display(dfs[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping.

In [ ]:
# Example: Filter and process a numeric field by its @id
# We'll look for a likely numeric field (e.g., age, diagnosis interval) in the columns

import numpy as np

df = None
if main_record_set_id is not None and main_record_set_id in dfs:
    df = dfs[main_record_set_id].copy()

numeric_field_id = None
possible_numeric_fields = ['age', 'interval', 'years', 'duration', 'metastasis']
for col in (df.columns if df is not None else []):
    # The columns are field @ids, but metadata may provide labels in 'name' or 'label'
    # We'll use @id for referencing as per instructions
    for key in possible_numeric_fields:
        if key in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id:
        break
        
if numeric_field_id is None:
    print("No obvious numeric field found in columns.")
else:
    # Convert field to numeric, coerce errors
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 0
    print(f"Filtering records where {numeric_field_id} > {threshold:.2f}")
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records (top 5):")
    display(filtered_df.head())

    # Normalize
    if filtered_df[numeric_field_id].std() != 0:
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to find a likely group/categorical field
    group_field_id = None
    possible_group_fields = ['sex', 'gender', 'msi', 'status', 'type', 'location', 'category']
    for col in df.columns:
        for key in possible_group_fields:
            if key in col.lower() and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            break
    if group_field_id and group_field_id in filtered_df.columns:
        # Only group by if column is not entirely NaN or empty
        if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped filtered data by {group_field_id} (showing mean {numeric_field_id}):")
            display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships in the main data record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group field is found, show boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 tabular dataset using `mlcroissant`:
- We reviewed record set and field `@id`s for referencing and reproducibility.
- Data from the Croissant schema was loaded, transformed, filtered, and briefly visualized.
- Key numeric and categorical variables were processed for early insights.

This template can be applied to other Croissant-structured datasets for fast, reproducible EDA using record/field `@id`s. For more details on this dataset, please refer to the metadata and associated documentation.